# 🚀 Обучение крипто-бота на бесплатном GPU (Kaggle)

Этот ноутбук настроен для обучения модели на **5 топовых криптовалютах** и **7 таймфреймах**.

**Важно:** Перед запуском убедитесь, что включен GPU:
1. Нажмите `Settings` (справа) → `Accelerator` → Выберите `GPU T4 x2`
2. Убедитесь, что включен `Internet` (нужен для загрузки данных с Binance)
3. Сохраните ноутбук (`Save Version`) для запуска в фоне

## 1. Установка зависимостей

Kaggle уже имеет предустановленные torch, pandas, numpy, scikit-learn. Установим только ccxt.

In [ ]:
!pip install ccxt yfinance --quiet
print("✅ Зависимости установлены: ccxt, yfinance")


## 2. Проверка окружения и настройка путей

В Kaggle данные сохраняются в `/kaggle/working/`, а модели можно выгрузить через `Dataset output`.

In [ ]:
import os
import torch
import pandas as pd
import numpy as np

# Проверка GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Используемое устройство: {device}")
if torch.cuda.is_available():
    print(f"📊 Видеокарта: {torch.cuda.get_device_name(0)}")
    print(f"💾 Доступно памяти: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Создание папок
os.makedirs('/kaggle/working/models', exist_ok=True)
os.makedirs('/kaggle/working/data', exist_ok=True)
os.makedirs('/kaggle/working/configs', exist_ok=True)
print("✅ Папки созданы")

## 3. Конфигурация

Настройка криптовалют, таймфреймов и параметров модели.

In [ ]:
# Конфигурация
CONFIG = {
    'symbols': ['BTC/USDT', 'ETH/USDT', 'BNB/USDT', 'SOL/USDT', 'XRP/USDT'],
    'timeframes': ['5m', '15m', '1h', '4h', '12h', '1d', '1w'],
    'model': {
        'hidden_size': 128,
        'num_layers': 2,
        'dropout': 0.2,
    },
    'training': {
        'epochs': 50,
        'batch_size': 32,
        'learning_rate': 0.001,
        'test_split': 0.2,
        'early_stopping_patience': 5,
    },
    'lookback_periods': {
        '5m': 200,
        '15m': 150,
        '1h': 100,
        '4h': 80,
        '12h': 60,
        '1d': 50,
        '1w': 30,
    }
}

print(f"📋 Криптовалюты: {len(CONFIG['symbols'])}")
print(f"⏱️ Таймфреймы: {len(CONFIG['timeframes'])}")
print(f"🧠 Параметры модели: {CONFIG['model']}")

## 4. Сбор данных с Binance

Функция для загрузки OHLCV данных через CCXT.

In [ ]:
# ============================================================
# ЗАГРУЗКА ДАННЫХ (YFINANCE ВМЕСТО BINANCE)
# ============================================================
# Binance блокирует IP адреса облачных сервисов (Kaggle/Colab) с ошибкой 451.
# Используем Yahoo Finance как надежный бесплатный источник.

import yfinance as yf
from datetime import datetime

print("📡 Использование источника данных: Yahoo Finance")

def fetch_crypto_data(symbol: str, timeframe: str, limit: int = 1000):
    """
    Загружает данные через yfinance.
    """
    # Маппинг таймфреймов для yfinance
    tf_map = {
        '5m': '5m',
        '15m': '15m',
        '1h': '1h',
        '4h': '1h',   # Агрегируем позже
        '12h': '1h',  # Агрегируем позже
        '1d': '1d',
        '1w': '1wk'
    }
    
    yf_interval = tf_map.get(timeframe, '1d')
    yf_symbol = symbol.replace('/', '-').replace('USDT', 'USD')
    
    try:
        ticker = yf.Ticker(yf_symbol)
        df = ticker.history(period='max', interval=yf_interval)
        
        if df.empty:
            return None
            
        # Агрегация для 4h и 12h
        if timeframe == '4h':
            df = df.resample('4h').agg({
                'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'
            }).dropna()
        elif timeframe == '12h':
            df = df.resample('12h').agg({
                'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'
            }).dropna()
            
        df = df.reset_index()
        df.columns = [c.lower() for c in df.columns]
        if 'date' in df.columns:
            df.rename(columns={'date': 'timestamp'}, inplace=True)
        elif 'datetime' in df.columns:
            df.rename(columns={'datetime': 'timestamp'}, inplace=True)
            
        df = df.tail(limit).reset_index(drop=True)
        return df
        
    except Exception as e:
        print(f"⚠️ Ошибка загрузки {symbol} ({timeframe}): {e}")
        return None

# Загрузка данных
all_data = {}

for symbol in CONFIG['symbols']:
    print(f"\n📥 Загрузка данных для {symbol}...")
    symbol_data = {}
    
    for tf in CONFIG['timeframes']:
        limit = CONFIG['lookback_periods'].get(tf, 1000) * 2
        df = fetch_crypto_data(symbol, tf, limit=limit)
        
        if df is not None and not df.empty:
            symbol_data[tf] = df
            print(f"   ✅ {tf}: загружено {len(df)} свечей")
        else:
            print(f"   ❌ {tf}: не удалось загрузить данные")
    
    if symbol_data:
        all_data[symbol] = symbol_data
    else:
        print(f"   ⚠️ Пропускаем {symbol}, нет данных")

print(f"\n✅ Загрузка завершена. Успешно загружено данных для {len(all_data)} криптовалют.")


## 5. Инжиниринг признаков

Добавление технических индикаторов: RSI, SMA, волатильность и др.

In [ ]:
def add_features(df):
    """
    Добавление технических индикаторов
    """
    df = df.copy()
    
    # RSI (Relative Strength Index)
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))
    
    # SMA (Simple Moving Average)
    df['sma_20'] = df['close'].rolling(window=20).mean()
    df['sma_50'] = df['close'].rolling(window=50).mean()
    df['sma_200'] = df['close'].rolling(window=200).mean()
    
    # EMA (Exponential Moving Average)
    df['ema_12'] = df['close'].ewm(span=12, adjust=False).mean()
    df['ema_26'] = df['close'].ewm(span=26, adjust=False).mean()
    
    # MACD
    df['macd'] = df['ema_12'] - df['ema_26']
    df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    
    # Bollinger Bands
    df['bb_middle'] = df['close'].rolling(window=20).mean()
    bb_std = df['close'].rolling(window=20).std()
    df['bb_upper'] = df['bb_middle'] + (bb_std * 2)
    df['bb_lower'] = df['bb_middle'] - (bb_std * 2)
    df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['bb_middle']
    
    # Volatility
    df['volatility'] = df['close'].pct_change().rolling(window=14).std()
    df['returns'] = df['close'].pct_change()
    
    # Volume indicators
    df['volume_sma'] = df['volume'].rolling(window=20).mean()
    df['volume_ratio'] = df['volume'] / df['volume_sma']
    
    # Price position
    df['price_position'] = (df['close'] - df['low']) / (df['high'] - df['low'])
    
    # Target: Следующее изменение цены (1 = рост, 0 = падение)
    df['target'] = (df['close'].shift(-1) > df['close']).astype(int)
    
    # Удаление NaN
    df.dropna(inplace=True)
    
    return df

# Применение к тестовым данным
if test_df is not None:
    test_featured = add_features(test_df)
    print(f"✅ Добавлено признаков: {len(test_featured.columns)}")
    print(f"\nПризнаки: {list(test_featured.columns)}")

## 6. Multi-Timeframe LSTM Модель

Архитектура с отдельными LSTM для каждого таймфрейма и attention механизмом.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class AttentionLayer(nn.Module):
    """
    Attention механизм для объединения скрытых состояний разных таймфреймов
    """
    def __init__(self, hidden_size):
        super(AttentionLayer, self).__init__()
        self.attention = nn.Linear(hidden_size, 1)
    
    def forward(self, lstm_outputs):
        # lstm_outputs: [batch_size, num_timeframes, hidden_size]
        attention_weights = F.softmax(self.attention(lstm_outputs), dim=1)
        context = torch.sum(attention_weights * lstm_outputs, dim=1)
        return context, attention_weights


class MultiTimeframeLSTM(nn.Module):
    """
    Multi-Timeframe LSTM с attention механизмом
    """
    def __init__(self, input_size, hidden_size, num_layers, dropout, num_timeframes):
        super(MultiTimeframeLSTM, self).__init__()
        
        self.num_timeframes = num_timeframes
        
        # Отдельный LSTM для каждого таймфрейма
        self.lstms = nn.ModuleList([
            nn.LSTM(input_size, hidden_size, num_layers, 
                   batch_first=True, dropout=dropout if num_layers > 1 else 0)
            for _ in range(num_timeframes)
        ])
        
        # Attention слой
        self.attention = AttentionLayer(hidden_size)
        
        # Финальный классификатор
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        # x: [batch_size, num_timeframes, sequence_length, input_size]
        batch_size = x.size(0)
        lstm_outputs = []
        
        for i in range(self.num_timeframes):
            lstm_out, _ = self.lstms[i](x[:, i, :, :])
            # Берем последний шаг времени
            lstm_outputs.append(lstm_out[:, -1, :])
        
        # Stack: [batch_size, num_timeframes, hidden_size]
        lstm_stack = torch.stack(lstm_outputs, dim=1)
        
        # Attention
        context, attention_weights = self.attention(lstm_stack)
        
        # Классификация
        output = self.fc(context)
        
        return output, attention_weights

# Тест модели
num_tf = len(CONFIG['timeframes'])
input_size = 17  # Количество признаков
test_model = MultiTimeframeLSTM(
    input_size=input_size,
    hidden_size=CONFIG['model']['hidden_size'],
    num_layers=CONFIG['model']['num_layers'],
    dropout=CONFIG['model']['dropout'],
    num_timeframes=num_tf
).to(device)

print(f"✅ Модель создана для {num_tf} таймфреймов")
print(f"📊 Вход: [batch, {num_tf}, seq_len, {input_size}]")
print(f"📈 Выход: [batch, 1]")
print(f"\n{test_model}")

## 7. Подготовка данных для обучения

Создание последовательностей для каждого таймфрейма.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

def prepare_multitimeframe_data(symbol_data_dict, lookback_periods, features):
    """
    Подготовка данных для multi-timeframe модели
    symbol_data_dict: dict {timeframe: DataFrame}
    """
    scalers = {}
    timeframe_sequences = {}
    
    # Масштабирование и создание последовательностей для каждого таймфрейма
    for tf, df in symbol_data_dict.items():
        scaler = MinMaxScaler()
        scaled_data = scaler.fit_transform(df[features])
        scalers[tf] = scaler
        
        lookback = lookback_periods.get(tf, 60)
        sequences = []
        targets = []
        
        for i in range(lookback, len(scaled_data) - 1):
            sequences.append(scaled_data[i-lookback:i])
            targets.append(df['target'].iloc[i])
        
        timeframe_sequences[tf] = {
            'sequences': np.array(sequences),
            'targets': np.array(targets),
            'lookback': lookback
        }
    
    # Синхронизация длин (берем минимальную длину среди всех таймфреймов)
    min_len = min(len(seq['sequences']) for seq in timeframe_sequences.values())
    
    # Объединение в единый тензор [samples, num_timeframes, lookback, features]
    X_list = []
    y_list = []
    
    timeframes = list(timeframe_sequences.keys())
    for i in range(min_len):
        sample = []
        for tf in timeframes:
            sample.append(timeframe_sequences[tf]['sequences'][i])
        X_list.append(sample)
        y_list.append(timeframe_sequences[timeframes[0]]['targets'][i])
    
    X = np.array(X_list)
    y = np.array(y_list)
    
    return X, y, scalers

print("✅ Функция подготовки данных готова")

## 8. Обучение модели

Полный цикл обучения для всех криптовалют.

In [ ]:
def train_all_models():
    """
    Обучение моделей для всех криптовалют
    """
    symbols = CONFIG['symbols']
    timeframes = CONFIG['timeframes']
    lookback_periods = CONFIG['lookback_periods']
    model_config = CONFIG['model']
    train_config = CONFIG['training']
    
    # Признаки
    features = [
        'open', 'high', 'low', 'close', 'volume',
        'rsi', 'sma_20', 'sma_50', 'sma_200',
        'ema_12', 'ema_26', 'macd', 'macd_signal',
        'bb_width', 'volatility', 'returns', 'volume_ratio'
    ]
    
    results = {}
    
    for symbol in symbols:
        print(f"\n{'='*60}")
        print(f"🚀 Обучение модели для {symbol}")
        print(f"{'='*60}")
        
        # 1. Сбор данных
        symbol_data = {}
        for tf in timeframes:
            df = fetch_data(symbol, tf, limit=1000)
            if df is not None:
                df = add_features(df)
                symbol_data[tf] = df
        
        if len(symbol_data) == 0:
            print(f"❌ Нет данных для {symbol}")
            continue
        
        # 2. Подготовка данных
        print("\n📊 Подготовка данных...")
        X, y, scalers = prepare_multitimeframe_data(
            symbol_data, lookback_periods, features
        )
        
        print(f"📦 Форма данных: {X.shape}")
        print(f"🎯 Целевая переменная: {y.shape}")
        
        # 3. Разделение на train/test
        split_idx = int(len(X) * (1 - train_config['test_split']))
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]
        
        print(f"📚 Train samples: {len(X_train)}, Test samples: {len(X_test)}")
        
        # 4. DataLoaders
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train),
            torch.FloatTensor(y_train).unsqueeze(1)
        )
        test_dataset = TensorDataset(
            torch.FloatTensor(X_test),
            torch.FloatTensor(y_test).unsqueeze(1)
        )
        
        train_loader = DataLoader(train_dataset, batch_size=train_config['batch_size'], shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=train_config['batch_size'])
        
        # 5. Модель
        num_timeframes = len(timeframes)
        model = MultiTimeframeLSTM(
            input_size=len(features),
            hidden_size=model_config['hidden_size'],
            num_layers=model_config['num_layers'],
            dropout=model_config['dropout'],
            num_timeframes=num_timeframes
        ).to(device)
        
        criterion = nn.BCELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=train_config['learning_rate'])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=3
        )
        
        # 6. Обучение
        best_loss = float('inf')
        patience_counter = 0
        
        for epoch in range(train_config['epochs']):
            # Train
            model.train()
            train_loss = 0.0
            
            for batch_X, batch_y in train_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)
                
                optimizer.zero_grad()
                output, _ = model(batch_X)
                loss = criterion(output, batch_y)
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
            
            avg_train_loss = train_loss / len(train_loader)
            
            # Validation
            model.eval()
            val_loss = 0.0
            correct = 0
            total = 0
            
            with torch.no_grad():
                for batch_X, batch_y in test_loader:
                    batch_X = batch_X.to(device)
                    batch_y = batch_y.to(device)
                    
                    output, _ = model(batch_X)
                    loss = criterion(output, batch_y)
                    val_loss += loss.item()
                    
                    predictions = (output > 0.5).float()
                    correct += (predictions == batch_y).sum().item()
                    total += batch_y.size(0)
            
            avg_val_loss = val_loss / len(test_loader)
            accuracy = correct / total
            
            scheduler.step(avg_val_loss)
            
            # Логирование
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"Epoch {epoch+1}/{train_config['epochs']} | "
                      f"Train Loss: {avg_train_loss:.4f} | "
                      f"Val Loss: {avg_val_loss:.4f} | "
                      f"Accuracy: {accuracy:.4f}")
            
            # Early stopping
            if avg_val_loss < best_loss:
                best_loss = avg_val_loss
                patience_counter = 0
                
                # Сохранение лучшей модели
                model_path = f"/kaggle/working/models/{symbol.replace('/', '_')}_model.pth"
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': best_loss,
                    'config': CONFIG,
                    'features': features,
                }, model_path)
                print(f"💾 Модель сохранена в {model_path}")
            else:
                patience_counter += 1
                if patience_counter >= train_config['early_stopping_patience']:
                    print(f"⏹️ Ранняя остановка на эпохе {epoch+1}")
                    break
        
        results[symbol] = {
            'best_loss': best_loss,
            'final_accuracy': accuracy,
            'model_path': model_path
        }
    
    # Итоговый отчет
    print(f"\n{'='*60}")
    print("🎉 ОБУЧЕНИЕ ЗАВЕРШЕНО!")
    print(f"{'='*60}")
    for symbol, res in results.items():
        print(f"{symbol}: Loss={res['best_loss']:.4f}, Acc={res['final_accuracy']:.4f}")
    
    return results

# Запуск обучения
print("🚀 Готов к запуску обучения!")
print("Нажмите Run All для начала или вызовите train_all_models()")

## 9. Запуск обучения

Выполните эту ячейку для старта обучения всех моделей.

In [ ]:
# Запуск обучения
results = train_all_models()

## 10. Сохранение результатов

Сохранение метрик и подготовка к выгрузке в Kaggle Dataset.

In [ ]:
import json

# Сохранение метрик обучения
metrics_path = '/kaggle/working/training_metrics.json'

# Конвертация для JSON serialization
serializable_results = {}
for symbol, res in results.items():
    serializable_results[symbol] = {
        'best_loss': float(res['best_loss']),
        'final_accuracy': float(res['final_accuracy']),
        'model_path': res['model_path']
    }

with open(metrics_path, 'w') as f:
    json.dump(serializable_results, f, indent=2)

print(f"✅ Метрики сохранены в {metrics_path}")

# Список файлов для выгрузки
print("\n📁 Файлы для выгрузки:")
for root, dirs, files in os.walk('/kaggle/working'):
    for file in files:
        filepath = os.path.join(root, file)
        filesize = os.path.getsize(filepath) / 1024 / 1024  # MB
        print(f"  {filepath} ({filesize:.2f} MB)")

print("\n💡 Для сохранения моделей:")
print("1. Нажмите 'Save Version' в правом верхнем углу")
print("2. Выберите 'Save & Run All (Commit)'")
print("3. После завершения в разделе 'Output' скачайте модели или создайте новый Dataset")